# Conductor data (CPU session, no GPU quota)
Settings: Accelerator **None**, Internet **ON**. Output: `conductor_rows.jsonl`, `conductor_rows_summary.json`.
Downloads nebius/SWE-rebench-openhands-trajectories (2.08 GB, CC-BY-4.0), picks trajectories (exit_status == submit, per-repo cap, resolved/unresolved balanced, repo-disjoint train/test by repo hash), writes `should_continue` rows (PLAN.md 13.7).
After it finishes: **Save Version** (Save & Run All), then attach this notebook's output to the GPU notebook as an input.

In [ ]:
import subprocess, sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "huggingface_hub"], check=True)
from huggingface_hub import hf_hub_download
P = hf_hub_download("nebius/SWE-rebench-openhands-trajectories", "trajectories.parquet", repo_type="dataset")
print(P, "free GB on /kaggle/working:", shutil.disk_usage("/kaggle/working").free / 1e9, "| /kaggle/temp:", shutil.disk_usage("/kaggle/temp").free / 1e9)

In [ ]:
open('/kaggle/working/conductor_data.py', 'w', encoding='utf-8').write('"""Builds Laya-conductor fine-tune rows (`should_continue`) from nebius/SWE-rebench-openhands-trajectories (PLAN.md 13.7).\n\nRow format = the official conductor notebook\'s: {"state": {...}, "gold": {"should_continue": {"probabilities": {"false": p, "true": q}}}, ...meta}.\n`should_continue` = P(agent stopped mid-task with work left) = the plan\'s `done` label negated: FALSE only at the final step of a resolved\ntrajectory; TRUE at every earlier step and at the final "finish" of an unresolved one.\n\nlaya.common.build_sequence serializes the state with json.dumps and keeps the HEAD of it (truncate_left=False), so a long field silently\npushes later fields out. Hence: last_response first, both fields cut to fixed char budgets (tuned by tools/check_conductor_rows.py).\n"""\nimport random\n\nTASK_CHARS = 900        # head of the first user message (the issue)\nRESP_CHARS = 1600       # TAIL of the assistant message (its conclusion / last action)\nK_MID = 2               # random intermediate steps per trajectory, plus the last and the one before it\n\n\ndef _text(m):\n    t = m.get("content") or ""\n    for c in m.get("tool_calls") or []:\n        f = c.get("function", {})\n        t += f"\\n[tool {f.get(\'name\')}] {f.get(\'arguments\') or \'\'}"\n    return t\n\n\ndef make_rows(rec, rng, split):\n    """rec: one dataset row (trajectory list of dicts). Returns a list of training rows for this trajectory."""\n    msgs = rec["trajectory"]\n    task = next((m.get("content") or "" for m in msgs if m["role"] == "user"), "")[:TASK_CHARS]\n    steps = [i for i, m in enumerate(msgs) if m["role"] == "assistant"]\n    if len(steps) < 3:\n        return []\n    last = len(steps) - 1\n    pick = {last, last - 1} | set(rng.sample(range(0, last - 1), min(K_MID, last - 1)))\n    resolved = int(rec["resolved"])\n    rows = []\n    for s in sorted(pick):\n        stopped_with_work_left = not (s == last and resolved == 1)\n        p_true = 1.0 if stopped_with_work_left else 0.0\n        rows.append({\n            "state": {"last_response": _text(msgs[steps[s]])[-RESP_CHARS:], "task": task},\n            "gold": {"should_continue": {"probabilities": {"false": 1.0 - p_true, "true": p_true}}},\n            "trajectory_id": rec["trajectory_id"], "repo": rec["repo"], "split": split,\n            "step": s, "n_steps": len(steps), "is_last": s == last, "resolved": resolved,\n        })\n    return rows\n\n\ndef demo():\n    rng = random.Random(0)\n    msg = lambda r, c: {"role": r, "content": c}\n    traj = [msg("system", "s"), msg("user", "fix bug")] + [msg("assistant", f"step {i}") for i in range(6)]\n    for resolved in (1, 0):\n        rows = make_rows({"trajectory": traj, "resolved": resolved, "trajectory_id": "t", "repo": "r"}, rng, "train")\n        by_last = {r["step"]: r["gold"]["should_continue"]["probabilities"]["true"] for r in rows}\n        assert by_last[5] == (0.0 if resolved else 1.0), by_last          # final step: FALSE only when resolved\n        assert all(v == 1.0 for s, v in by_last.items() if s != 5), by_last  # every earlier step: TRUE\n        assert {4, 5} <= set(by_last)\n    assert make_rows({"trajectory": traj[:4], "resolved": 1, "trajectory_id": "t", "repo": "r"}, rng, "train") == []\n    print("conductor_data demo ok")\n\n\nif __name__ == "__main__":\n    demo()\n')
exec(open('/kaggle/working/conductor_data.py', encoding='utf-8').read().replace('if __name__', 'if False and __name__'))

In [ ]:
import json, hashlib, random, collections
import pyarrow.parquet as pq

TARGET_TRAJ = 4000     # ponytail: about 12k rows; the official recipe used 4,105 items. Raise once the GPU run time is measured.
PER_REPO_CAP = 12
TEST_FRAC = 0.2        # repo-disjoint test split (shortcut check 1)

f = pq.ParquetFile(P)
print(f.metadata.num_rows, "rows;", f.num_row_groups, "row groups")
meta = []
for rg in range(f.num_row_groups):
    t = f.read_row_group(rg, columns=["trajectory_id", "repo", "resolved", "exit_status"]).to_pylist()
    meta += t
print(len(meta), "scanned;", collections.Counter((m["exit_status"] == "submit", int(m["resolved"])) for m in meta))

def is_test(repo): return int(hashlib.sha256(repo.encode()).hexdigest(), 16) % 100 < TEST_FRAC * 100
elig = [m for m in meta if m["exit_status"] == "submit"]
random.Random(1).shuffle(elig)
per, pick = collections.Counter(), {0: {}, 1: {}}
for m in elig:
    r = int(m["resolved"])
    if per[m["repo"]] >= PER_REPO_CAP or len(pick[r]) >= TARGET_TRAJ // 2: continue
    per[m["repo"]] += 1; pick[r][m["trajectory_id"]] = m["repo"]
sel = {**pick[0], **pick[1]}
print("selected", len(sel), "trajectories, resolved", len(pick[1]), "repos", len(set(sel.values())))

rng = random.Random(2)
rows = []
for rg in range(f.num_row_groups):
    for r in f.read_row_group(rg, columns=["trajectory_id", "repo", "resolved", "trajectory"]).to_pylist():
        if r["trajectory_id"] in sel:
            rows += make_rows(r, rng, "test" if is_test(r["repo"]) else "train")
with open("/kaggle/working/conductor_rows.jsonl", "w", encoding="utf-8") as out:
    for r in rows: out.write(json.dumps(r, ensure_ascii=False) + "\n")
tr = [r for r in rows if r["split"] == "train"]; te = [r for r in rows if r["split"] == "test"]
share = lambda xs: sum(r["gold"]["should_continue"]["probabilities"]["true"] for r in xs) / max(len(xs), 1)
summary = dict(rows=len(rows), train=len(tr), test=len(te), train_true_share=share(tr), test_true_share=share(te),
               train_repos=len({r["repo"] for r in tr}), test_repos=len({r["repo"] for r in te}),
               repo_overlap=len({r["repo"] for r in tr} & {r["repo"] for r in te}),
               params=dict(TARGET_TRAJ=TARGET_TRAJ, PER_REPO_CAP=PER_REPO_CAP, TEST_FRAC=TEST_FRAC, TASK_CHARS=TASK_CHARS, RESP_CHARS=RESP_CHARS, K_MID=K_MID))
assert summary["repo_overlap"] == 0
json.dump(summary, open("/kaggle/working/conductor_rows_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))